# Training CNN-1D URL Classifier & Export ke TFLite
**Google Colab | Multi-Dataset | Sequential CNN-1D | Float16 TFLite**

---
Notebook ini melatih model CNN-1D berbasis karakter untuk mengklasifikasikan URL
sebagai **aman** (0) atau **pornografi** (1), kemudian mengekspornya ke format TFLite
yang siap dijalankan di aplikasi Android.

**Arsitektur Sequential CNN-1D:**
```
Input  [1, MAX_LEN]  dtype=INT32  ← array indeks karakter
  │
  └── Embedding(vocab=41, dim=32) → shape [1, MAX_LEN, 32]
  │
  └── Conv1D(num_filters, kernel_size, relu)
  │
  └── MaxPooling1D
  │
  └── Dropout
  │
  └── GlobalMaxPooling1D
  │
  └── Dense(64, relu)
  │
  └── Dense(1, sigmoid)

Output [1, 1]  dtype=FLOAT32  ← probabilitas porno (0.0–1.0)
```

**Alur kerja:**
1. Load CSV per ukuran dataset dari `DATASET_FILES`
2. Sampling balanced + tokenisasi full domain
3. Training dengan early stopping (monitor val_loss)
4. Evaluasi (accuracy, F1, confusion matrix, ROC)
5. Export ke TFLite float16 + simpan ke Drive

## Cell 1 — Install Library

In [ ]:
!pip install -q --upgrade scikit-learn seaborn
print('Install selesai.')

## Cell 2 — Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import re
import time
import shutil
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, roc_curve, auc, classification_report
)

print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU        : {gpus if gpus else "Tidak tersedia — pastikan Runtime > T4 GPU"}')

# Seed global untuk reprodusibilitas
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Cell 3 — Mount Google Drive & Konfigurasi

**Format dataset yang diharapkan** — CSV dengan minimal 2 kolom:
```
url, label
```
atau jika menggunakan dataset RF yang sama:
```
url, label, domain_length, digit_count, ...
```

> `label`: 0 = URL aman, 1 = URL pornografi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ================================================================
# SESUAIKAN PATH INI
# ================================================================
DATASET_BASE     = '/content/drive/MyDrive/Tugas Akhir/Dataset/'
SAVE_PATH        = '/content/drive/MyDrive/Tugas Akhir/Training/CNN1D/'
BEST_PARAMS_JSON = '/content/drive/MyDrive/Tugas Akhir/Tuning/CNN1D/best_params_cnn.json'
URL_COL   = 'url'
LABEL_COL = 'label'

# Daftar file CSV — setiap file = satu ukuran dataset terpisah
# SIZE_LABEL otomatis dari nama file (tanpa ekstensi), misal:
#   dataset_100rb.csv  →  SIZE_LABEL = 'dataset_100rb'
# Tambah/hapus baris di sini untuk mengatur ukuran yang ingin dilatih
DATASET_FILES = [
    DATASET_BASE + 'dataset_100rb.csv',
    DATASET_BASE + 'dataset_200rb.csv',
    DATASET_BASE + 'dataset_300rb.csv',
]
# ================================================================

os.makedirs(SAVE_PATH, exist_ok=True)
print('Dataset files:')
for fpath in DATASET_FILES:
    size_label = os.path.splitext(os.path.basename(fpath))[0]
    ok = os.path.exists(fpath)
    if ok:
        tmp = pd.read_csv(fpath, usecols=[LABEL_COL])
        n0 = int((tmp[LABEL_COL] == 0).sum())
        n1 = int((tmp[LABEL_COL] == 1).sum())
        print(f'  {size_label}: OK  {len(tmp):,} baris | aman={n0:,} | porno={n1:,}')
    else:
        print(f'  {size_label}: FILE TIDAK ADA — upload dulu ke Drive')

## Cell 4 — Fungsi Tokenisasi (IDENTIK dengan Android)

> **KRITIS**: Seluruh kode di cell ini harus 1:1 sama dengan `UrlClassifier.kt`.
> Perbedaan sekecil apapun akan menyebabkan model membaca input yang salah di Android.

### Pemetaan Karakter (CHAR_TO_IDX)
| Range | Indeks | Karakter |
|-------|--------|----------|
| PAD   | 0      | `\u0000` (null / padding) |
| UNK   | 1      | Semua karakter tidak dikenal |
| a–z   | 2–27   | 26 huruf alfabet |
| 0–9   | 28–37  | 10 digit angka |
| .     | 38     | Titik |
| -     | 39     | Tanda hubung |
| _     | 40     | Garis bawah |

**Vocab size = 41** (indeks 0 sampai 40)

In [ ]:
# ============================================================
# CHAR_TO_IDX — identik dengan Android UrlClassifier.kt
# Urutan harus PERSIS sama:
#   put('\u0000', 0)  → chr(0)  = PAD
#   for a..z         → a=2, b=3, ..., z=27
#   for 0..9         → 0=28, 1=29, ..., 9=37
#   '.'=38, '-'=39, '_'=40
# ============================================================
CHAR_TO_IDX: dict = {chr(0): 0}  # PAD
for _i, _c in enumerate('abcdefghijklmnopqrstuvwxyz'):
    CHAR_TO_IDX[_c] = _i + 2       # a=2, ..., z=27
for _i, _c in enumerate('0123456789'):
    CHAR_TO_IDX[_c] = _i + 28      # 0=28, ..., 9=37
CHAR_TO_IDX['.'] = 38
CHAR_TO_IDX['-'] = 39
CHAR_TO_IDX['_'] = 40

PAD_IDX    = 0
UNK_IDX    = 1
VOCAB_SIZE = 41  # 0–40

# TLD yang dikenal — identik dengan COMMON_TLDS di UrlClassifier.kt
COMMON_TLDS = {
    'com','net','org','info','biz','name','pro','int',
    'co','io','me','tv','cc','ws','in','ru','cn','jp','kr',
    'de','uk','fr','it','es','br','au','ca','nl','be','ch',
    'at','pl','se','no','dk','fi','cz','hu','ro','bg','gr',
    'pt','ie','nz','za','sg','hk','tw','my','th','ph','id','vn',
    'xyz','top','site','online','club','live','fun','space',
    'tech','store','shop','app','dev','cloud','digital','media',
    'news','blog','video','games','world','network','global',
    'center','zone','today','one','life','work','money','email',
    'link','click','download','stream','watch','porn','sex',
    'xxx','adult','cam','tube','how',
    'tk','ml','ga','cf','gq',
    'ltd','vip','pw','asia','mobi','tel','travel','jobs','edu','gov','mil'
}

SECOND_LEVEL_TLDS = {
    'co.id','co.uk','co.jp','co.kr','co.nz','co.za','co.in','co.th',
    'com.au','com.br','com.cn','com.hk','com.my','com.sg','com.tw',
    'com.vn','com.ph','com.ar','com.mx','com.co','com.pe','com.ve',
    'com.ec','com.pk','com.bd','com.ng','com.eg','com.tr','com.ua','com.ru',
    'net.au','net.br','net.cn','net.id','net.in','net.nz','net.za',
    'org.au','org.br','org.cn','org.id','org.in','org.nz','org.uk','org.za',
    'ac.id','ac.uk','ac.jp','ac.kr','ac.nz','ac.za','ac.th',
    'edu.au','edu.br','edu.cn','edu.hk','edu.my','edu.sg','edu.tw','edu.vn',
    'go.id','go.jp','go.kr','go.th',
    'or.id','or.jp','or.kr','or.th',
    'ne.jp','ne.kr',
    'web.id','sch.id','my.id','biz.id'
}


def normalize_domain(url: str) -> str:
    """Normalisasi URL → domain — identik dengan Android normalizeDomain()"""
    url = url.lower().strip()
    for prefix in ['https://', 'http://', 'www.']:
        if url.startswith(prefix):
            url = url[len(prefix):]
    url = url.split('/')[0].split('?')[0].split('#')[0].split(':')[0]
    return url


def extract_main_domain_name(full_domain: str) -> str:
    """Ekstrak nama inti domain — identik dengan Android extractMainDomainName()"""
    parts = full_domain.split('.')
    if len(parts) < 2:
        return full_domain
    if len(parts) >= 3:
        potential_2nd = f'{parts[-2]}.{parts[-1]}'
        if potential_2nd in SECOND_LEVEL_TLDS:
            return parts[-3] if len(parts) >= 3 else parts[-2]
    if parts[-1] in COMMON_TLDS:
        return parts[-2]
    return parts[-2] if len(parts) >= 2 else full_domain


def tokenize(domain: str, max_len: int) -> list:
    """Ubah string domain ke array indeks karakter — identik dengan Android tokenize()"""
    tokens = [PAD_IDX] * max_len
    for i, char in enumerate(domain[:max_len]):
        tokens[i] = CHAR_TO_IDX.get(char, UNK_IDX)
    return tokens


# ============================================================
# VERIFIKASI — pastikan output identik dengan Android
# ============================================================
print('Verifikasi pemetaan CHAR_TO_IDX:')
print(f'  a={CHAR_TO_IDX["a"]} (exp=2), z={CHAR_TO_IDX["z"]} (exp=27)')
print(f'  0={CHAR_TO_IDX["0"]} (exp=28), 9={CHAR_TO_IDX["9"]} (exp=37)')
print(f'  .={CHAR_TO_IDX["."]} (exp=38), -={CHAR_TO_IDX["-"]} (exp=39), _={CHAR_TO_IDX["_"]} (exp=40)')
print(f'  VOCAB_SIZE={VOCAB_SIZE} (exp=41)')
print()

print('Verifikasi tokenisasi:')
tests = [
    ('https://www.youporn.com/watch', 'youporn.com', 'youporn'),
    ('http://xvideos.com/video123', 'xvideos.com', 'xvideos'),
    ('https://mail.google.com/inbox', 'mail.google.com', 'google'),
    ('https://pornhub.com', 'pornhub.com', 'pornhub'),
]
print(f'{"URL":<40} {"Domain":<20} {"Core":<15} Tokens[0:8]')
print('-' * 90)
for url, exp_domain, exp_core in tests:
    d = normalize_domain(url)
    core = extract_main_domain_name(d)
    tok = tokenize(core, 34)
    ok_d = '✅' if d == exp_domain else f'❌ (exp:{exp_domain})'
    ok_c = '✅' if core == exp_core else f'❌ (exp:{exp_core})'
    print(f'{url:<40} {d:<20} {core:<15} {tok[:8]}  dom:{ok_d} core:{ok_c}')

## Cell 5 — Load Hyperparameter Terbaik

Jika file `best_params_cnn.json` tidak ada (belum menjalankan tuning),
notebook ini menggunakan **default hyperparameter** yang sudah reasonable.

In [ ]:
# Default hyperparameter — digunakan jika belum melakukan tuning
# (sesuai Tabel III.3, nilai tengah dari setiap rentang)
DEFAULT_PARAMS = {
    'num_filters'  : 64,
    'kernel_size'  : 5,
    'dropout_rate' : 0.2,
    'learning_rate': 1e-3,
    'embed_dim'    : 32,   # tetap, tidak dituning
    'dense_units'  : 64,   # tetap, tidak dituning
}

if os.path.exists(BEST_PARAMS_JSON):
    with open(BEST_PARAMS_JSON) as f:
        params = json.load(f)
    print(f'Loaded dari: {BEST_PARAMS_JSON}')
else:
    params = DEFAULT_PARAMS
    print(f'best_params_cnn.json tidak ditemukan.')
    print(f'   Menggunakan default hyperparameter.')
    print(f'   Jalankan cnn1d_hyperparameter_tuning.ipynb untuk tuning.')

# Ambil nilai dengan fallback ke default
NUM_FILTERS   = int(params.get('num_filters',   DEFAULT_PARAMS['num_filters']))
KERNEL_SIZE   = int(params.get('kernel_size',   DEFAULT_PARAMS['kernel_size']))
DROPOUT_RATE  = float(params.get('dropout_rate',  DEFAULT_PARAMS['dropout_rate']))
LEARNING_RATE = float(params.get('learning_rate', DEFAULT_PARAMS['learning_rate']))
EMBED_DIM     = int(params.get('embed_dim',     DEFAULT_PARAMS['embed_dim']))
DENSE_UNITS   = int(params.get('dense_units',   DEFAULT_PARAMS['dense_units']))

print(f'\nHyperparameter yang digunakan:')
print(f'  num_filters   : {NUM_FILTERS}')
print(f'  kernel_size   : {KERNEL_SIZE}')
print(f'  dropout_rate  : {DROPOUT_RATE}')
print(f'  learning_rate : {LEARNING_RATE}')
print(f'  embed_dim     : {EMBED_DIM}  (fixed)')
print(f'  dense_units   : {DENSE_UNITS}  (fixed)')
print(f'  vocab_size    : {VOCAB_SIZE}')
print(f'  (MAX_LEN dihitung per dataset di dalam training loop)')

## Cell 6 — Bangun Model Sequential CNN-1D

**Detail arsitektur (sesuai laporan Subbab II.1.9):**

1. **Embedding layer**: mengubah indeks karakter (int32) menjadi vektor padat (float32)
   - Input: `[batch, MAX_LEN]` dtype INT32
   - Output: `[batch, MAX_LEN, embed_dim]`

2. **Conv1D**: ekstraksi pola n-gram dari urutan karakter URL
   - `kernel_size=3`: trigram seperti `sex`, `xxx`, `cam`
   - `kernel_size=5`: pentagram seperti `video`, `adult`
   - `kernel_size=7`: heptagram seperti `youporn`, `xvideos`
   - Nilai kernel_size ditentukan dari tuning (default: 5)

3. **MaxPooling1D**: down-sampling, pertahankan fitur dominan

4. **Dropout**: regularisasi, cegah overfitting

5. **GlobalMaxPooling1D**: ambil nilai fitur paling signifikan dari seluruh feature map

6. **Dense (Fully Connected)**: klasifikasi berdasarkan fitur yang diekstrak

7. **Output sigmoid**: probabilitas URL porno [0.0, 1.0]
   - ≥ 0.7: diklasifikasikan sebagai porno (threshold di Android)
   - < 0.7: diklasifikasikan sebagai aman

> **Kenapa INT32 input?** Android menggunakan `buffer.putInt(token)` untuk model
> dengan Embedding layer. TFLite mempertahankan dtype INT32 meski menggunakan
> float16 quantization untuk weights.

In [ ]:
def build_cnn1d_model(
    vocab_size    : int,
    max_len       : int,
    embed_dim     : int,
    num_filters   : int,
    kernel_size   : int,
    dense_units   : int,
    dropout_rate  : float,
    learning_rate : float
) -> tf.keras.Model:
    """
    Sequential CNN-1D untuk klasifikasi URL berbasis karakter.
    Arsitektur: Embedding → Conv1D → MaxPool → Dropout
               → GlobalMaxPool → Dense → Output(Sigmoid)
    Input dtype INT32 agar kompatibel dengan Android (TFLite INT32 → Embedding).
    """
    # Input: array indeks karakter [batch, max_len] dtype=int32
    inputs = tf.keras.Input(shape=(max_len,), dtype='int32', name='input')

    # Embedding: int32 → float32 vektor padat
    x = layers.Embedding(
        input_dim    = vocab_size,
        output_dim   = embed_dim,
        input_length = max_len,
        name         = 'embedding'
    )(inputs)

    # Conv1D: ekstraksi pola n-gram karakter URL
    x = layers.Conv1D(
        filters     = num_filters,
        kernel_size = kernel_size,
        activation  = 'relu',
        padding     = 'same',
        name        = 'conv1d'
    )(x)

    # MaxPooling1D: down-sampling, pertahankan fitur dominan
    x = layers.MaxPooling1D(name='maxpool')(x)

    # Dropout: regularisasi, cegah overfitting
    x = layers.Dropout(dropout_rate, name='dropout')(x)

    # GlobalMaxPooling1D: ambil nilai fitur paling signifikan dari seluruh feature map
    x = layers.GlobalMaxPooling1D(name='global_maxpool')(x)

    # Dense: Fully Connected Layer untuk klasifikasi
    x = layers.Dense(dense_units, activation='relu', name='dense')(x)

    # Output: probabilitas porno, range [0.0, 1.0]
    output = layers.Dense(1, activation='sigmoid', name='output')(x)

    model = Model(inputs=inputs, outputs=output, name='CNN1D_URL_Classifier')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ]
    )
    return model


print('Fungsi build_cnn1d_model() siap. Model akan dibangun di dalam training loop.')

## Cell 7 — Training Loop (Multi-Dataset)

Loop ini memproses setiap file CSV di `DATASET_FILES` secara berurutan:
1. Load & balanced sampling per kelas
2. Preprocessing & tokenisasi full domain (tanpa augmentasi)
3. Split 70/15/15 → bangun model → training dengan EarlyStopping
4. Evaluasi test set (accuracy, F1, AUC, confusion matrix)
5. Export TFLite float16 → simpan ke Drive

Estimasi waktu per dataset (GPU T4 Colab): **5–15 menit**

In [ ]:
# ── Konfigurasi Global ───────────────────────────────────────────
BATCH_SIZE = 256
MAX_EPOCHS = 50
THRESHOLD  = 0.7

all_results = {}  # hasil semua ukuran untuk perbandingan akhir

for csv_path in DATASET_FILES:
    # SIZE_LABEL otomatis dari nama file, misal dataset_100rb.csv -> 'dataset_100rb'
    SIZE_LABEL = os.path.splitext(os.path.basename(csv_path))[0]

    # Load CSV untuk ukuran ini
    df_cur  = pd.read_csv(csv_path)
    n_aman  = int((df_cur[LABEL_COL] == 0).sum())
    n_porno = int((df_cur[LABEL_COL] == 1).sum())
    N_PER_CLASS = min(n_aman, n_porno)   # balanced: ambil semua data per kelas

    print(f'\n{"="*65}')
    print(f'  {SIZE_LABEL}  ({len(df_cur):,} baris | {N_PER_CLASS:,}/kelas)')
    print(f'{"="*65}')

    SIZE_SAVE_PATH = SAVE_PATH + f'{SIZE_LABEL}/'
    os.makedirs(SIZE_SAVE_PATH, exist_ok=True)

    # ── 1. Sampling balanced ─────────────────────────────────────
    df_safe = df_cur[df_cur[LABEL_COL] == 0]
    df_porn = df_cur[df_cur[LABEL_COL] == 1]
    df_bal  = pd.concat([
        df_safe.sample(n=N_PER_CLASS, random_state=SEED),
        df_porn.sample(n=N_PER_CLASS, random_state=SEED)
    ]).sample(frac=1, random_state=SEED).reset_index(drop=True)

    # ── 2. Preprocessing ─────────────────────────────────────────
    df_bal['full_domain'] = df_bal[URL_COL].apply(lambda u: normalize_domain(str(u)))
    df_bal = df_bal[
        df_bal['full_domain'].str.contains(r'\.', regex=True) &
        df_bal['full_domain'].str.len().gt(3)
    ].reset_index(drop=True)

    # MAX_LEN dari P95 panjang full domain
    all_lengths = df_bal['full_domain'].str.len()
    MAX_LEN = int(np.percentile(all_lengths, 95))
    MAX_LEN = max(MAX_LEN, 20); MAX_LEN = min(MAX_LEN, 100)

    # ── 3. Tokenisasi (full domain saja, tanpa augmentasi) ────────
    X_list, y_list = [], []
    for _, row in df_bal.iterrows():
        label       = int(row[LABEL_COL])
        full_domain = str(row['full_domain'])
        if full_domain and len(full_domain) >= 4:
            X_list.append(tokenize(full_domain, MAX_LEN))
            y_list.append(label)

    X = np.array(X_list, dtype=np.int32)
    y = np.array(y_list, dtype=np.float32)
    print(f'Total sampel: {len(X):,} | MAX_LEN: {MAX_LEN}')
    print(f'Distribusi: aman={int((y==0).sum()):,}, porno={int((y==1).sum()):,}')

    # ── 4. Split 70/15/15 ────────────────────────────────────────
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.15, random_state=SEED, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.176, random_state=SEED, stratify=y_temp
    )
    print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

    # ── 5. Bangun Model ──────────────────────────────────────────
    tf.keras.backend.clear_session()
    model = build_cnn1d_model(
        vocab_size=VOCAB_SIZE, max_len=MAX_LEN, embed_dim=EMBED_DIM,
        num_filters=NUM_FILTERS, kernel_size=KERNEL_SIZE, dense_units=DENSE_UNITS,
        dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE
    )

    checkpoint_path = f'/content/cnn1d_best_{SIZE_LABEL}.keras'
    callbacks_run = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.CSVLogger(f'/content/training_log_{SIZE_LABEL}.csv'),
    ]

    # ── 6. Training ───────────────────────────────────────────────
    print(f'\nTraining {SIZE_LABEL}...')
    t0 = time.time()
    history = model.fit(
        X_train, y_train, validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE, epochs=MAX_EPOCHS,
        callbacks=callbacks_run, verbose=1
    )
    durasi       = time.time() - t0
    n_epochs_run = len(history.history['loss'])
    best_val_acc = max(history.history['val_accuracy'])
    best_ep      = history.history['val_accuracy'].index(best_val_acc) + 1
    print(f'Selesai: {durasi/60:.1f} menit | best epoch: {best_ep} | val_acc: {best_val_acc:.4f}')

    # ── 7. Plot Training History ──────────────────────────────────
    hist_h   = history.history
    ep_range = range(1, len(hist_h['loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'Training History — CNN-1D {SIZE_LABEL}', fontsize=12)
    axes[0].plot(ep_range, hist_h['loss'],     label='Train')
    axes[0].plot(ep_range, hist_h['val_loss'], label='Val', linestyle='--')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(ep_range, hist_h['accuracy'],     label='Train')
    axes[1].plot(ep_range, hist_h['val_accuracy'], label='Val', linestyle='--')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'/content/training_history_{SIZE_LABEL}.png', dpi=120, bbox_inches='tight')
    plt.show(); plt.close()

    # ── 8. Evaluasi Test Set ──────────────────────────────────────
    y_pred_prob = model.predict(X_test, batch_size=512, verbose=0).flatten()
    y_pred      = (y_pred_prob >= THRESHOLD).astype(int)
    y_test_int  = y_test.astype(int)
    acc  = accuracy_score(y_test_int, y_pred)
    prec = precision_score(y_test_int, y_pred, zero_division=0)
    rec  = recall_score(y_test_int, y_pred, zero_division=0)
    f1   = f1_score(y_test_int, y_pred, zero_division=0)
    fpr, tpr, _ = roc_curve(y_test_int, y_pred_prob)
    roc_auc = auc(fpr, tpr)

    print(f'\nTest Set — {SIZE_LABEL}:')
    print(f'  Accuracy: {acc:.4f}  Precision: {prec:.4f}')
    print(f'  Recall  : {rec:.4f}  F1-Score : {f1:.4f}  AUC: {roc_auc:.4f}')
    print(classification_report(y_test_int, y_pred,
          target_names=['Aman (0)', 'Pornografi (1)'], digits=4))

    # ── 9. Confusion Matrix + Distribusi Skor ────────────────────
    cm = confusion_matrix(y_test_int, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Evaluasi — CNN-1D {SIZE_LABEL}', fontsize=12)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Pred: Aman', 'Pred: Porno'],
                yticklabels=['Actual: Aman', 'Actual: Porno'], ax=axes[0])
    axes[0].set_title('Confusion Matrix')
    axes[1].hist(y_pred_prob[y_test == 0], bins=50, alpha=0.6,
                 label='Aman', color='green', density=True)
    axes[1].hist(y_pred_prob[y_test == 1], bins=50, alpha=0.6,
                 label='Porno', color='red', density=True)
    axes[1].axvline(THRESHOLD, color='black', linestyle='--',
                    label=f'Thr={THRESHOLD}')
    axes[1].set_title('Distribusi Skor'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'/content/evaluation_{SIZE_LABEL}.png', dpi=120, bbox_inches='tight')
    plt.show(); plt.close()

    # ── 10. Export TFLite ─────────────────────────────────────────
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    try:
        tflite_model = converter.convert()
    except Exception:
        converter2 = tf.lite.TFLiteConverter.from_keras_model(model)
        tflite_model = converter2.convert()
    tflite_path = f'/content/CNN1D_{SIZE_LABEL}.tflite'
    with open(tflite_path, 'wb') as f_out:
        f_out.write(tflite_model)
    print(f'TFLite: {len(tflite_model)/1024:.1f} KB → {tflite_path}')

    # ── 11. Simpan ke Drive ───────────────────────────────────────
    for src, fname in [
        (tflite_path,                                    f'CNN1D_{SIZE_LABEL}.tflite'),
        (checkpoint_path,                                f'cnn1d_model_{SIZE_LABEL}.keras'),
        (f'/content/training_history_{SIZE_LABEL}.png',  f'training_history_{SIZE_LABEL}.png'),
        (f'/content/evaluation_{SIZE_LABEL}.png',        f'evaluation_{SIZE_LABEL}.png'),
        (f'/content/training_log_{SIZE_LABEL}.csv',      f'training_log_{SIZE_LABEL}.csv'),
    ]:
        if os.path.exists(src):
            shutil.copy(src, SIZE_SAVE_PATH + fname)
            print(f'  ✅ {fname}')

    # ── 12. Catat hasil ───────────────────────────────────────────
    all_results[SIZE_LABEL] = {
        'csv_file'         : os.path.basename(csv_path),
        'total_rows_csv'   : len(df_cur),
        'n_per_class'      : N_PER_CLASS,
        'total_samples'    : len(X),
        'max_len'          : MAX_LEN,
        'best_epoch'       : best_ep,
        'n_epochs_run'     : n_epochs_run,
        'best_val_accuracy': round(best_val_acc, 4),
        'train_time_min'   : round(durasi / 60, 1),
        'accuracy'         : round(acc,     4),
        'precision'        : round(prec,    4),
        'recall'           : round(rec,     4),
        'f1_score'         : round(f1,      4),
        'auc_roc'          : round(roc_auc, 4),
        'tflite_kb'        : round(len(tflite_model) / 1024, 1),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
    }
    print(f'\n✅ SELESAI {SIZE_LABEL} — acc={acc:.4f} | f1={f1:.4f} | {len(tflite_model)/1024:.1f}KB')
    print(f'{"─"*65}')

# ── Simpan semua hasil ke JSON ────────────────────────────────────
with open(SAVE_PATH + 'cnn1d_all_results.json', 'w') as f_out:
    json.dump(all_results, f_out, indent=2)
print(f'\n✅ all_results: {SAVE_PATH}cnn1d_all_results.json')

# ── Ringkasan Perbandingan Semua Ukuran ──────────────────────────
print(f'\n{"="*65}')
print('SEMUA TRAINING SELESAI — RINGKASAN')
print(f'{"="*65}')
print(f'{"Dataset":<20} {"Rows":>8} {"Samples":>10} {"Acc":>7} {"F1":>7} {"AUC":>7} {"KB":>7}')
print(f'{"─"*65}')
for lbl, r in all_results.items():
    print(f'{lbl:<20} {r["total_rows_csv"]:>8,} {r["total_samples"]:>10,} '
          f'{r["accuracy"]:>7.4f} {r["f1_score"]:>7.4f} {r["auc_roc"]:>7.4f} {r["tflite_kb"]:>7.1f}')
print(f'{"="*65}')